# Project 2 — Fraud Detection

**Goal:** Train and evaluate a classifier on synthetic imbalanced data. This is not real banking data.

Run the cells from top to bottom. Data is generated locally or loaded from scikit-learn; no CSV download is required.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, silhouette_score, mean_absolute_error, mean_squared_error,
    precision_score, recall_score
)
from sklearn.datasets import load_breast_cancer, make_classification, make_blobs
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
print("Setup complete")


## Project 2 — Fraud Detection (Imbalanced Classification)
**Goal:** Train a classifier and evaluate it using precision, recall, F1, and ROC-AUC. The dataset is synthetic; it is not real banking data.

In [ ]:
X_fraud, y_fraud = make_classification(
    n_samples=5000, n_features=20, n_informative=8, n_redundant=4,
    weights=[0.97, 0.03], flip_y=0.01, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X_fraud, y_fraud, test_size=0.25, stratify=y_fraud, random_state=RANDOM_STATE
)

fraud_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
)
fraud_model.fit(X_train, y_train)
pred = fraud_model.predict(X_test)
proba = fraud_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, target_names=["Normal", "Potential fraud"]))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Normal", "Potential fraud"])
plt.title("Fraud detection confusion matrix")
plt.show()

In [ ]:
# Explore how changing the alert threshold changes precision/recall
from sklearn.metrics import precision_score, recall_score

threshold_rows = []
for threshold in [0.20, 0.35, 0.50, 0.65, 0.80]:
    pred_t = (proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0)
    })
display(pd.DataFrame(threshold_rows).round(3))

## Possible extensions
- Compare Random Forest and gradient boosting.
- Tune the alert threshold based on false-positive and false-negative costs.